In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1070").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/04 03:21:11 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.103 instead (on interface enp0s3)
25/08/04 03:21:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/04 03:21:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Sales

+-------------+-------+
| Column Name | Type  |
+-------------+-------+
| sale_id     | int   |
| product_id  | int   |
| year        | int   |
| quantity    | int   |
| price       | int   |
+-------------+-------+
(sale_id, year) is the primary key (combination of columns with unique values) of this table.
product_id is a foreign key (reference column) to Product table.
Each row of this table shows a sale on the product product_id in a certain year.
Note that the price is per unit.
 

Table: Product

+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| product_id   | int     |
| product_name | varchar |
+--------------+---------+
product_id is the primary key (column with unique values) of this table.
Each row of this table indicates the product name of each product.
 

Write a solution to select the product id, year, quantity, and price for the first year of every product sold.

Return the resulting table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Sales table:
+---------+------------+------+----------+-------+
| sale_id | product_id | year | quantity | price |
+---------+------------+------+----------+-------+ 
| 1       | 100        | 2008 | 10       | 5000  |
| 2       | 100        | 2009 | 12       | 5000  |
| 7       | 200        | 2011 | 15       | 9000  |
+---------+------------+------+----------+-------+
Product table:
+------------+--------------+
| product_id | product_name |
+------------+--------------+
| 100        | Nokia        |
| 200        | Apple        |
| 300        | Samsung      |
+------------+--------------+
Output: 
+------------+------------+----------+-------+
| product_id | first_year | quantity | price |
+------------+------------+----------+-------+ 
| 100        | 2008       | 10       | 5000  |
| 200        | 2011       | 15       | 9000  |
+------------+------------+----------+-------+
'''

In [2]:
sales_data = [
(1,100,2008,10,5000),
(2,100,2009,12,5000),
(7,200,2011,15,9000)
]
sales_schema = ['sale_id','product_id','year','quantity','price']

product_data = [
(100,'Nokia'),
(200,'Apple'),
(300,'Samsung')
]
product_schema = ['product_id','product_name']

In [3]:
sales_df = spark.createDataFrame(data = sales_data, schema = sales_schema)
product_df = spark.createDataFrame(data = product_data, schema = product_schema)

In [7]:
first_year_products_df = sales_df.groupBy(F.col("product_id"))\
                  .agg(F.min(F.col("year")).alias("year"))

first_year_products_df.alias("t").join(sales_df.alias("t1"), 
                        (F.col("t1.product_id") == F.col("t.product_id")) & 
                        (F.col("t1.year") == F.col("t.year")),
                        'left'
                       ).select(
                        F.col("t1.product_id"),F.col("t1.year").alias("first_year"),
                        F.col("t1.quantity"),F.col("t1.price")
                               )\
                        .show()

+----------+----------+--------+-----+
|product_id|first_year|quantity|price|
+----------+----------+--------+-----+
|       100|      2008|      10| 5000|
|       200|      2011|      15| 9000|
+----------+----------+--------+-----+



## SQL Solution

<pre>
WITH TEMP AS (
SELECT product_id, MIN(year) as year 
FROM Sales 
GROUP BY product_id)
SELECT t1.product_id, t1.year as first_year, t1.quantity, t1.price
FROM temp t
LEFT JOIN Sales t1 ON t.product_id = t1.product_id 
                   AND t.year = t1.year
    
</pre>